# Representation augmentation — can we fix the resource-notation brittleness?

E5 showed the released CE model transfers to real tool-call *vocabulary* but is
sensitive to resource *notation*: on real tau2 it scores **90.8%** in the trained
`family:namespace/leaf` (slash) notation and **75.0%** when the same data is
re-rendered in an all-colon notation. This notebook tests whether training across
notations fixes that, and reports the answer honestly either way.

**Three-way comparison, all on the same held-out real tests:**
1. **released** — the single-notation CE model (baseline, no retraining).
2. **balanced-mix** — CE retrained with an even 4-way notation mix
   (`canonical`/`allcolon`/`allslash`/`pipe`). *Finding: this backfired* — it
   equalized the notations by dragging slash **down** to colon's level
   (~75%/~72%) and made the model **over-authorize** real redirects
   (false-authorize ~50%), i.e. it overfit the augmented synthetic corpus.
3. **canonical-majority** — CE retrained keeping the trained slash notation as
   the **70% majority**, the other 30% split across the other schemes, so
   resource discrimination stays sharp while gaining some notation tolerance.
   *Hypothesis: slash recovers toward ~90% and colon rises above the balanced
   ~72%.*

**Data hygiene.** train = expanded corpus (seed 101), val = expanded corpus
(seed **202**, deduplicated vs train and the committed test); both augmented.
Held-out real tests, never seen in training, from two independent logs:
tau2 (`Jarrodbarnes/tau2-sft-v4`, balanced confused-deputy) in **both** slash and
colon notation, and Toucan (`Agent-Ark/Toucan-1.5M`, a second independent MCP
corpus) authorized-only in **mixed native notations**. Every label comes from our
verifier.

Runtime → T4/L4 → Run all. Training is 2 variants × 3 seeds (~2.5–3 h); eval uses
a robust in-process runner that prints a traceback on any failure instead of
silently dropping a row. Download `augment_results.zip` at the end.

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
!git pull --ff-only
import torch; assert torch.cuda.is_available()
!PYTHONPATH=. python test_authority_verifier.py
!PYTHONPATH=. python make_expanded_train.py --train-seed 101 --train-traces-per-class 150 --val-seed 202 --val-traces-per-class 25

## Step 1 — build the augmented corpora (balanced + canonical-majority) and tests

In [ ]:
# balanced 4-way mix (the negative baseline)
!PYTHONPATH=. python -u augment_representation.py --in expanded_train.jsonl --out augmented_train.jsonl --seed 33
!PYTHONPATH=. python -u augment_representation.py --in expanded_val.jsonl   --out augmented_val.jsonl   --seed 34
# canonical-majority mix (70% trained slash, 30% split across other notations)
!PYTHONPATH=. python -u augment_representation.py --in expanded_train.jsonl --out augmented_train_cmaj.jsonl --seed 33 --canonical-frac 0.7
!PYTHONPATH=. python -u augment_representation.py --in expanded_val.jsonl   --out augmented_val_cmaj.jsonl   --seed 34 --canonical-frac 0.7
!PYTHONPATH=. python test_augment_representation.py
!PYTHONPATH=. python test_map_toucan.py

## Step 2 — train CE for both variants, 3 seeds each

`balanced` → `ckpt_balanced_seed{7,8,9}`, `cmaj` → `ckpt_cmaj_seed{7,8,9}`.
Each monitors on its own augmented val. `python -u` streams live.

In [ ]:
# balanced 4-way mix
!for s in 7 8 9; do echo "=== balanced seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --ce-loss --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file augmented_train.jsonl --test-file augmented_val.jsonl --eval-every 50 --eval-max-actions 120 --seed $s --log-file training_log_balanced_seed$s.jsonl --save-dir ckpt_balanced_seed$s || break; done
# canonical-majority mix
!for s in 7 8 9; do echo "=== cmaj seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --ce-loss --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file augmented_train_cmaj.jsonl --test-file augmented_val_cmaj.jsonl --eval-every 50 --eval-max-actions 120 --seed $s --log-file training_log_cmaj_seed$s.jsonl --save-dir ckpt_cmaj_seed$s || break; done

## Step 3 — build the held-out REAL tests (never seen in training)

Two independent real logs. **tau2** (balanced confused-deputy) in both the
trained slash notation and the naive colon notation. **Toucan** (second,
independent source; authorized-only) rendered in **mixed native notations**
via `augment()` — each real trace gets a random delimiter scheme, labels
re-verified. All labels come from our verifier.

In [ ]:
from trace_benchmark import load_traces, write_jsonl
from augment_representation import renotate_trace, augment

# tau2 — balanced confused-deputy, slash (trained) + colon (naive)
!PYTHONPATH=. python map_tau_to_chain.py --seed 5   # writes real_trace_*.jsonl (slash)
slash = load_traces('real_trace_all.jsonl')
write_jsonl([renotate_trace(t, 'allcolon') for t in slash], 'real_trace_all_colon.jsonl')
print('tau2  slash:', slash[0]['actions'][0]['resource'],
      '| colon:', renotate_trace(slash[0], 'allcolon')['actions'][0]['resource'])

# Toucan — independent real source, authorized-only, MIXED native notations
!PYTHONPATH=. python map_toucan_to_chain.py --shards 1 --max-traj 200 --seed 5
toucan = load_traces('real_toucan_all.jsonl')
mixed, discarded = augment(toucan, seed=9)   # random delimiter scheme per trace, labels re-verified
write_jsonl(mixed, 'real_toucan_mixed.jsonl')
from collections import Counter
print('toucan mixed notations:', dict(Counter(t['_notation'] for t in mixed)),
      '| discarded (label changed):', discarded, '| n_traces:', len(mixed))

## Step 4 — evaluate released vs balanced vs canonical-majority on every held-out test

Robust in-process runner: each (model, test) is one subprocess; a non-zero exit
prints the stderr tail instead of silently dropping the row (the bug that hid the
real-test rows in the first run). tau2 is balanced (accuracy + false-authorize
matter); Toucan is authorized-only (its metric is false-refuse).

In [ ]:
import json
import os
import subprocess

RESULTS = 'results_augment.json'
json.dump({'backends': {}}, open(RESULTS, 'w'))

released = 'esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b'
real = ['real_trace_all.jsonl', 'real_trace_all_colon.jsonl', 'real_toucan_mixed.jsonl']

# released baseline (no retraining), then both augmented variants across seeds
jobs = [(released, ['benchmark_test.jsonl'] + real)]
for tag in ('balanced', 'cmaj'):
    for s in (7, 8, 9):
        ck = 'ckpt_%s_seed%d' % (tag, s)
        if os.path.isdir(ck):
            jobs.append((ck, ['benchmark_test.jsonl'] + real))

env = dict(os.environ, PYTHONPATH='.')
for model, files in jobs:
    for tf in files:
        if not os.path.exists(tf):
            print('SKIP', model, '::', tf, '(missing)')
            continue
        cmd = ['python', 'train_verifier_reward.py', '--eval-checkpoint',
               model, '--test-file', tf, '--merge-results', RESULTS]
        r = subprocess.run(cmd, env=env, capture_output=True, text=True)
        if r.returncode != 0:
            print('FAIL', model, '::', tf)
            print(r.stderr[-2000:])
        else:
            print('ok  ', model, '::', tf)

In [ ]:
import json
from stats import wilson_ci

d = json.load(open('results_augment.json'))
def pc(x):
    return 'n/a' if x is None else format(100 * x, '5.1f')
print('%-56s %4s %20s %6s %6s' % ('checkpoint / test', 'n', 'acc% [95% CI]', 'fauth', 'fref'))
for name in sorted(d['backends']):
    m = d['backends'][name]['metrics']
    n = m['n_actions']
    c = wilson_ci(round(m['accuracy'] * n), n)
    ci = '%5.1f [%.1f,%.1f]' % (100 * m['accuracy'], 100 * c[0], 100 * c[1])
    print('%-56s %4d %20s %6s %6s' % (
        name.replace('local:', '')[:56], n, ci,
        pc(m['false_authorize_rate']), pc(m['false_refuse_rate'])))
print('')
print('tau2 = balanced (watch acc + fauth); Toucan = authorized-only (watch fref).')
print('WIN if cmaj recovers tau2-slash toward ~90% AND lifts tau2-colon above ~72%.')

In [ ]:
!zip -j augment_results.zip training_log_balanced_*.jsonl training_log_cmaj_*.jsonl results_augment.json real_toucan_mixed.jsonl real_trace_all_colon.jsonl
from google.colab import files
files.download('augment_results.zip')